# GPGPU Tasks 2026 — Kaggle Runner (CUDA)

**Settings справа:** Accelerator = **GPU T4 x2** (или P100), Internet = **on**.

Workflow: правишь код локально → `git push` в ветку `task03` → здесь ячейка 4 (pull + make + запуск).

In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. [1 раз за сессию] Клонирование + зависимости
BRANCH = "task03"
%cd /kaggle/working
!rm -rf /kaggle/working/GPGPUTasks2026
!git clone -b $BRANCH https://github.com/BOGOMOLOV-ARSENIQ/GPGPUTasks2026.git /kaggle/working/GPGPUTasks2026
!sudo apt-get update -qq && sudo apt-get install -yqq unzip libgtest-dev libgraphicsmagick++1-dev

# На Ubuntu 24.04 пакета libspirv-reflect-dev нет (он есть только с 26.04),
# а проекту он не нужен (spirv_reflect лежит в libs/gpu) - выкидываем его из apt install скрипта
!sed -i 's/ libspirv-reflect-dev//' /kaggle/working/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh

# Ставим Vulkan SDK в чистой папке, полный лог - в /kaggle/working/vulkan_install.log
!rm -rf /tmp/vk && mkdir -p /tmp/vk && cd /tmp/vk && bash /kaggle/working/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh --no-validation-layers > /kaggle/working/vulkan_install.log 2>&1 || (tail -40 /kaggle/working/vulkan_install.log; echo "VULKAN INSTALL FAILED")

import os
assert os.path.exists("/usr/local/include/vulkan/vulkan.h"), "Vulkan не установился - смотри лог выше"
print("Vulkan SDK установлен")

In [ ]:
# 3. [1 раз за сессию] Конфигурация CMake (75 = T4)
# На Kaggle nvcc не лежит в PATH, и старый FindCUDA его не находит - ищем тулкит руками
import glob, os, subprocess
cands = [d for d in ["/usr/local/cuda"] + sorted(glob.glob("/usr/local/cuda-*"), reverse=True) if os.path.exists(d + "/bin/nvcc")]
assert cands, "CUDA toolkit (nvcc) не найден в /usr/local/cuda*"
CUDA_ROOT = cands[0]
os.environ["PATH"] = CUDA_ROOT + "/bin:" + os.environ["PATH"]
print("CUDA toolkit:", CUDA_ROOT)
!nvcc --version | tail -2

# libcuda.so ставится вместе с драйвером, т.е. есть только когда к сессии подключен GPU
ldconf = subprocess.run("ldconfig -p", shell=True, capture_output=True, text=True).stdout
libcuda = [l.split("=>")[-1].strip() for l in ldconf.splitlines() if "libcuda.so" in l and "x86-64" in l]
libcuda += glob.glob("/usr/local/nvidia/lib64/libcuda.so*") + glob.glob(CUDA_ROOT + "/lib64/stubs/libcuda.so")
assert libcuda, "libcuda.so не найден - GPU не подключен? Settings -> Accelerator -> GPU T4 x2, потом заново ячейки 1-3"
CUDA_DRIVER_LIB = libcuda[0]
print("CUDA driver lib:", CUDA_DRIVER_LIB)

!rm -rf /kaggle/working/GPGPUTasks2026/build && mkdir -p /kaggle/working/GPGPUTasks2026/build
%cd /kaggle/working/GPGPUTasks2026/build
!cmake -DCMAKE_BUILD_TYPE=Release -DGPU_CUDA_SUPPORT=ON -DCUDA_TOOLKIT_ROOT_DIR={CUDA_ROOT} -DCMAKE_CUDA_COMPILER={CUDA_ROOT}/bin/nvcc -DCUDA_cuda_driver_LIBRARY={CUDA_DRIVER_LIB} -DCMAKE_CUDA_ARCHITECTURES=75 ..

In [ ]:
# 4. [Каждый раз после git push] Подтянуть код, пересобрать, запустить
%cd /kaggle/working/GPGPUTasks2026
!git fetch -q origin && git reset -q --hard origin/$BRANCH && git log --oneline -1
%cd /kaggle/working/GPGPUTasks2026/build
!make -j$(nproc) 2>&1 | tail -20

print("\n\033[94m=== main_matrix_transpose ===\033[0m")
!./main_matrix_transpose 0
# CPU-эталон умножения идёт ~80 сек
print("\n\033[94m=== main_matrix_multiply ===\033[0m")
!./main_matrix_multiply 0

In [ ]:
# 4b. (опционально) Прогнать несколько версий подряд для сравнения на одной карте
# b139e82 = тайл 16x16, d343120 = тайл 32x32, origin/task03 = последняя версия (тайл 32x32, группа 32x8) (naive 8x32 есть в обеих - заодно видно разброс замеров)
for rev in ["b139e82", "d343120", "origin/task03"]:
    print(f"\n\033[94m=== {rev} ===\033[0m")
    !cd /kaggle/working/GPGPUTasks2026 && git checkout -q {rev} && git log --oneline -1
    !cd /kaggle/working/GPGPUTasks2026/build && make -j$(nproc) main_matrix_transpose 2>&1 | tail -2 && ./main_matrix_transpose 0
!cd /kaggle/working/GPGPUTasks2026 && git checkout -q $BRANCH